# Frequent itemsets on the episode transactions

**Pipeline:** `episodes_v1` → transactions → **frequent itemsets (this notebook)** → association rules → `recurring_cohorts.csv`.

**Goal:** implement Apriori, check it against the definition and against mlxtend, measure how the output and the cost grow as minsup falls, and choose minsup.

**Definitions**

- A **transaction** is the set of students in one episode; an **itemset** $X$ is a set of students.
- The **support count** $\sigma(X)$ is the number of episodes that contain all of $X$.
- **minsup** is a number of episodes; $X$ is **frequent** if $\sigma(X)\ge$ minsup.
- **Downward closure:** if $X\subseteq Y$ then $\sigma(X)\ge\sigma(Y)$, so every subset of a frequent itemset is frequent.

**Apriori, level by level**

1. Count the single students and keep the frequent ones ($L_1$).
2. **Join:** two frequent $(k-1)$-itemsets with the same first $k-2$ items form a candidate of size $k$.
3. **Prune:** drop a candidate if any of its $(k-1)$-subsets is not frequent.
4. **Count:** scan the transactions once and count the remaining candidates.
5. Keep the frequent candidates ($L_k$); stop when none are left.

Itemsets are stored as sorted tuples, so that the join can compare their prefixes.

In [ ]:
from itertools import combinations
from pathlib import Path
import random
import sys
import time
import pandas as pd
from mlxtend.frequent_patterns import apriori as mlx_apriori, fpgrowth, fpmax

PROJECT_ROOT = Path.cwd().parent                   # notebooks/ is one level below the project root
PROCESSED = PROJECT_ROOT / "data" / "processed"

sys.path.append(str(PROJECT_ROOT))                 # so "from src..." works inside notebooks/
from src.features.transactions import build_transactions, support_count, transaction_stats, to_one_hot

## 1. Functions

- `count_items`: the support of every single student.
- `make_candidates`: the join and the prune; it reads no data.
- `count_candidates`: one scan that counts the candidates.
- `apriori`: runs the levels and records, per level, the candidates before and after pruning and the frequent itemsets.

In [ ]:
def count_items(transactions):
    """Level 1: in how many transactions does each single item appear?"""
    counts = {}
    for transaction in transactions:
        for item in transaction:
            counts[(item,)] = counts.get((item,), 0) + 1  # 1-tuple key, same format as later levels; 0 on first sight
    return counts


def make_candidates(frequent_prev, k):
    """Join and prune: build the size-k candidates from the sorted list of frequent (k-1)-itemsets."""
    known = set(frequent_prev)                          # a set makes the prune lookups fast

    # join: same first k-2 items, and the last item of a is smaller than the last item of b
    joined = []
    for a in frequent_prev:
        for b in frequent_prev:
            if a[:-1] == b[:-1] and a[-1] < b[-1]:      # same prefix; "<" builds each candidate only once
                joined.append(a + (b[-1],))             # (1,2) + (1,3) gives (1,2,3)

    # prune: every (k-1)-subset of a candidate must be frequent
    candidates = []
    for candidate in joined:
        subsets = combinations(candidate, k - 1)        # (2,3,4) gives (2,3), (2,4), (3,4)
        if all(subset in known for subset in subsets):  # one infrequent subset kills the candidate
            candidates.append(candidate)
    return joined, candidates


def count_candidates(transactions, candidates, k):
    """One scan: for every transaction, look at its k-subsets and count the ones that are candidates."""
    candidate_set = set(candidates)                      # a set makes the subset lookups fast
    counts = {candidate: 0 for candidate in candidates}  # start at 0, so unseen candidates still appear
    for transaction in transactions:
        if len(transaction) < k:                         # too short to contain a k-itemset
            continue
        for subset in combinations(transaction, k):      # sorted transaction gives sorted subsets, like the candidates
            if subset in candidate_set:                  # once per transaction, so it counts distinct episodes
                counts[subset] += 1
    return counts


def apriori(transactions, minsup):
    """Return the list of frequent itemsets per level ({itemset: support}) and a table of candidate counts."""
    transactions = [tuple(sorted(t)) for t in transactions]                               # sorted tuples: the join compares prefixes

    item_counts = count_items(transactions)                                               # level 1: the first scan counts every item
    frequent = {itemset: n for itemset, n in item_counts.items() if n >= minsup}          # filter: support >= minsup
    levels = [frequent]                                                                   # levels[0] is L1
    rows = [{"k": 1, "joined": len(item_counts), "candidates": len(item_counts), "frequent": len(frequent)}]  # numbers per level for the report

    k = 2
    while len(frequent) > 0:                                                              # stop at an empty level (downward closure)
        keep = set(item for itemset in frequent for item in itemset)                      # items inside some frequent (k-1)-itemset
        transactions = [tuple(item for item in t if item in keep) for t in transactions]  # drop the others: fewer subsets to count

        joined, candidates = make_candidates(sorted(frequent), k)                         # join + prune, no data is read
        if len(candidates) > 0:                                                           # scan only if a candidate survived the prune
            counts = count_candidates(transactions, candidates, k)
        else:
            counts = {}
        frequent = {itemset: n for itemset, n in counts.items() if n >= minsup}           # filter after counting

        rows.append({"k": k, "joined": len(joined), "candidates": len(candidates), "frequent": len(frequent)})  # numbers of this level
        if len(frequent) > 0:                                                             # store non-empty levels only
            levels.append(frequent)                                                       # so len(levels) is the longest itemset size
        k += 1
    return levels, pd.DataFrame(rows)

## 2. Toy example

Seven transactions over the items A to E, minsup 3. Expected supports: A 5, B 6, C 5, AB 4, AC 4, BC 4, ABC 3. D (2) and E (1) are not frequent, so no candidate contains them.

In [ ]:
toy = [
    {"A", "B", "C"},        # T1
    {"A", "B", "C"},        # T2
    {"A", "B"},             # T3
    {"B", "C"},             # T4
    {"B", "D"},             # T5
    {"A", "B", "C", "D"},   # T6
    {"A", "C", "E"},        # T7
]
toy_levels, toy_table = apriori(toy, minsup=3)
print(toy_table.to_string(index=False))

expected = {("A",): 5, ("B",): 6, ("C",): 5,
            ("A", "B"): 4, ("A", "C"): 4, ("B", "C"): 4,
            ("A", "B", "C"): 3}
found = {itemset: n for level in toy_levels for itemset, n in level.items()}
print(found)
assert found == expected

## 3. Real transactions

One transaction per episode. A transaction with $m$ students has $2^m-1$ non-empty subsets, so the few very large episodes dominate the cost at low minsup.

In [ ]:
episodes = pd.read_parquet(PROCESSED / "episodes_v1.parquet")
transactions = build_transactions(episodes)
stats = transaction_stats(transactions)
stats

## 4. Candidates per level

Our Apriori at minsup 400 to 100. Per level: `joined` (after the join), `candidates` (after the prune, counted) and `frequent`. Minsup 75 did not finish within 45 seconds, so the list stops at 100.

In [ ]:
MINSUPS = [400, 300, 250, 200, 150, 100]            # counts of episodes, high to low

summary = []
tables = {}
results = {}
for minsup in MINSUPS:
    start = time.time()
    levels, table = apriori(transactions, minsup)
    seconds = time.time() - start

    tables[minsup] = table
    results[minsup] = levels
    summary.append({
        "minsup": minsup,
        "minsup_share_pct": round(100 * minsup / len(transactions), 2),
        "counted_candidates": int(table["candidates"].sum()),
        "frequent_itemsets": int(table["frequent"].sum()),
        "longest_itemset": len(levels),                # levels only holds non-empty levels
        "seconds": round(seconds, 1),
    })
    print(f"minsup = {minsup}")
    print(table.to_string(index=False))
    print()

pd.DataFrame(summary)

## 5. Checks

1. A random sample of the itemsets is recounted with `support_count` from `transactions.py`.
2. At level 2 nothing can be pruned, so the number of counted pairs must equal $\binom{|L_1|}{2}$.

In [ ]:
random.seed(0)
for minsup in MINSUPS:
    found = [(itemset, n) for level in results[minsup] for itemset, n in level.items()]
    for itemset, n in random.sample(found, min(30, len(found))):
        assert support_count(transactions, itemset) == n
print("sampled supports agree with the definition")

for minsup in MINSUPS:
    table = tables[minsup]
    n_l1 = int(table.loc[table["k"] == 1, "frequent"].iloc[0])
    n_c2 = int(table.loc[table["k"] == 2, "candidates"].iloc[0])
    assert n_c2 == n_l1 * (n_l1 - 1) // 2
    print(f"minsup {minsup}: |L1| = {n_l1}, |C2| = {n_c2} = C({n_l1}, 2)")

## 6. Library versions: mlxtend `apriori`, `fpgrowth` and `fpmax`

- The input is a True/False table, one row per episode and one column per student (`to_one_hot`).
- `min_support` is a fraction. We pass `(minsup − 0.5) / N`, because `minsup / N` can fall on the wrong side of the limit by rounding (413 instead of 417 itemsets in a test). Supports are converted back with `round(support * N)`.
- `apriori` needs `low_memory=True` at low minsup.
- `fpgrowth` returns the same itemsets as `apriori`; `fpmax` returns only the **maximal** ones (no frequent superset).
- `find_maximal` marks every subset one item smaller of a frequent itemset as not maximal; by downward closure this is enough.

In [ ]:
def find_maximal(levels):
    """Maximal itemsets: frequent itemsets that have no frequent superset."""
    not_maximal = set()                                             # itemsets that have a frequent superset
    for level in levels[1:]:                                        # itemsets of size 2 and up
        for itemset in level:
            for subset in combinations(itemset, len(itemset) - 1):  # its subsets with one item less
                not_maximal.add(subset)                             # they have a frequent superset (this itemset), so they are not maximal
    maximal = set()
    for level in levels:
        for itemset in level:
            if itemset not in not_maximal:                          # no frequent superset found
                maximal.add(itemset)
    return maximal


def levels_to_counts(levels):
    """Flatten the levels of our Apriori into one dictionary {itemset: support}."""
    counts = {}
    for level in levels:
        counts.update(level)  # add the itemsets of this level
    return counts


def mlxtend_to_counts(result, n_total):
    """Turn an mlxtend result (support as a fraction) into {sorted tuple: support count}."""
    counts = {}
    for itemset, support in zip(result["itemsets"], result["support"]):
        key = tuple(sorted(int(item) for item in itemset))  # same format as our itemsets
        counts[key] = round(support * n_total)              # from a fraction back to a count of episodes
    return counts


episode_toy = [{1, 2, 3}, {1, 2, 3, 4}, {1, 2, 3}, {2, 4, 5}, {1, 5, 6}]  # the toy from tests/test_transactions.py
episode_toy_levels, _ = apriori(episode_toy, minsup=2)                    # our Apriori at minsup 2
episode_toy_maximal = find_maximal(episode_toy_levels)
print(sorted(episode_toy_maximal))
assert episode_toy_maximal == {(1, 2, 3), (2, 4), (5,)}                   # maximal by hand: 123, 24 and 5

episode_toy_onehot = to_one_hot(episode_toy)                              # True/False table, one column per item
episode_toy_share = (2 - 0.5) / len(episode_toy)                          # a fraction just below 2/5
episode_toy_mlx = mlx_apriori(episode_toy_onehot, min_support=episode_toy_share, use_colnames=True)
assert mlxtend_to_counts(episode_toy_mlx, len(episode_toy)) == levels_to_counts(episode_toy_levels)  # same 10 itemsets and supports
print(len(episode_toy_mlx), "itemsets from mlxtend, the same as ours")

## 7. Runtime

Each library algorithm runs only where it finishes in reasonable time; a blank means not run. Times depend on the computer and the mlxtend version, so compare the columns, not the absolute values.

In [ ]:
onehot = to_one_hot(transactions)  # one row per episode, one column per participant (True/False)
n_episodes = len(transactions)
print(n_episodes, "episodes,", onehot.shape[1], "participants,", round(onehot.memory_usage().sum() / 1e6), "MB")

**Run the library algorithms** and record their times. The results are reused in sections 8 and 9.

In [ ]:
LIB_MINSUPS = [400, 300, 250, 200, 150, 100, 75, 50, 40]              # counts of episodes, high to low
LIB_RUN = {                                                           # where each library algorithm is run
    "apriori": LIB_MINSUPS,
    "fpgrowth": [400, 300, 250, 200, 150],
    "fpmax": [400, 300, 250, 200, 150, 100],
}
scratch_seconds = {row["minsup"]: row["seconds"] for row in summary}  # runtimes of our Apriori from section 4


def run_mlxtend(name, minsup):
    share = (minsup - 0.5) / n_episodes  # a fraction just below minsup / n_episodes, avoids float rounding
    if name == "apriori":
        return mlx_apriori(onehot, min_support=share, use_colnames=True, low_memory=True)
    if name == "fpgrowth":
        return fpgrowth(onehot, min_support=share, use_colnames=True)
    return fpmax(onehot, min_support=share, use_colnames=True)


lib_rows = []                                                                          # one row per minsup for the runtime table
lib_counts = {}                                                                        # {(name, minsup): {itemset: support}} for apriori and fpgrowth
fpmax_sets = {}                                                                        # {minsup: set of maximal itemsets from fpmax}

for minsup in LIB_MINSUPS:
    row = {"minsup": minsup, "share_pct": round(100 * minsup / n_episodes, 2)}
    if minsup in scratch_seconds:                                                      # our Apriori was run at the minsups of section 4
        row["scratch_s"] = scratch_seconds[minsup]

    for name in ["apriori", "fpgrowth", "fpmax"]:
        if minsup not in LIB_RUN[name]:                                                # too slow here, skip
            continue
        start = time.time()
        result = run_mlxtend(name, minsup)
        row[name + "_s"] = round(time.time() - start, 1)
        row[name + "_itemsets"] = len(result)
        if minsup in MINSUPS:                                                          # keep the result only where we have our own to compare with
            if name == "fpmax":
                fpmax_sets[minsup] = set(mlxtend_to_counts(result, n_episodes))        # fpmax: only which itemsets matter
            else:
                lib_counts[(name, minsup)] = mlxtend_to_counts(result, n_episodes)

    lib_rows.append(row)
    print(row)

runtime = pd.DataFrame(lib_rows).rename(columns={"apriori_itemsets": "frequent_itemsets", "fpmax_itemsets": "maximal_itemsets"})
runtime = runtime[["minsup", "share_pct", "scratch_s", "apriori_s", "fpgrowth_s", "fpmax_s", "frequent_itemsets", "maximal_itemsets"]]
runtime = runtime.astype({"frequent_itemsets": "Int64", "maximal_itemsets": "Int64"})  # whole numbers, blanks stay blank
runtime

**Plot of the runtimes.** One line per algorithm. Both axes are on a log scale and minsup falls from left to right. A line stops where that algorithm was not run.

In [ ]:
import matplotlib.pyplot as plt

algorithms = {                                                    # column in runtime: (label, colour, marker)
    "scratch_s": ("our Apriori", "#2a78d6", "o"),
    "apriori_s": ("mlxtend apriori", "#eb6834", "s"),
    "fpgrowth_s": ("mlxtend fpgrowth", "#1baf7a", "^"),
    "fpmax_s": ("mlxtend fpmax", "#eda100", "D"),
}

fig, ax = plt.subplots(figsize=(9, 5))
for column, (label, color, marker) in algorithms.items():
    seconds = runtime[column].astype(float).clip(lower=0.05)      # 0.0 s means "below 0.1 s", a log axis cannot show 0
    ax.plot(runtime["minsup"], seconds, color=color, marker=marker, lw=2, label=label)   # NaN = not run, the line stops
    last = runtime[column].last_valid_index()                     # last minsup where this algorithm was run
    ax.annotate(f"{runtime[column][last]} s", (runtime["minsup"][last], seconds[last]),
                textcoords="offset points", xytext=(8, 0), va="center")   # time at its last minsup

ax.set_xscale("log")                                              # equal ratios get equal distances, 50 and 40 stay readable
ax.set_yscale("log")                                              # the times differ by orders of magnitude
ax.invert_xaxis()                                                 # minsup falls from left to right
ax.set_xticks(runtime["minsup"])
ax.set_xticklabels(runtime["minsup"])
ax.minorticks_off()                                               # no extra tick labels on the log axes
ax.set_xlabel("minsup (number of episodes)")
ax.set_ylabel("runtime in seconds (log scale)")
ax.set_title("Runtime as minsup falls")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.show()

**Reading the runtimes**

- All algorithms slow down as minsup falls. At minsup 100, mlxtend `apriori` takes 0.8 s and our Apriori 8.6 s.
- In the stored run, `fpgrowth` and `fpmax` were slower than `apriori` (26.7 s and 7.8 s at minsup 150, against 0.2 s). This compares mlxtend's implementations, not the algorithms: a re-run with mlxtend 0.25 on another computer took 1.3 s for `fpgrowth`.
- In the table, `frequent_itemsets` comes from `apriori` and `maximal_itemsets` from `fpmax`.

## 8. Do the results agree?

For every minsup from 400 to 100, `apriori` and `fpgrowth` must match our itemsets and supports exactly, and `fpmax` must match `find_maximal`.

In [ ]:
for minsup in MINSUPS:                                                                       # the minsups where we have our own results
    ours = levels_to_counts(results[minsup])                                                 # {itemset: support} from section 4
    parts = [f"{len(ours)} frequent itemsets, ours"]

    for name in ["apriori", "fpgrowth"]:
        if (name, minsup) in lib_counts:                                                     # fpgrowth was not run at every minsup
            assert lib_counts[(name, minsup)] == ours, f"{name} differs at minsup {minsup}"  # same itemsets, same supports
            parts.append(name)

    maximal = find_maximal(results[minsup])                                                  # maximal itemsets computed from our frequent ones
    assert fpmax_sets[minsup] == maximal, f"fpmax differs at minsup {minsup}"

    print(f"minsup {minsup}: " + " = ".join(parts) + f"; {len(maximal)} maximal itemsets = fpmax")

**Result:** identical at every minsup and level. This covers only these minsups; the check against the definition is in section 5.

## 9. Itemsets as minsup falls

Counts from our own results: single students, itemsets of two or more students, maximal itemsets and closed itemsets.

In [ ]:
count_rows = []
for minsup in MINSUPS:                                              # the minsups where we have our own results
    levels = results[minsup]                                        # levels[0] = single items, levels[1] = pairs, and so on
    count_rows.append({
        "minsup": minsup,
        "share_pct": round(100 * minsup / n_episodes, 2),           # minsup as a percentage of all episodes
        "singles": len(levels[0]),                                  # frequent single participants
        "size_2_or_more": sum(len(level) for level in levels[1:]),  # frequent itemsets with at least two participants
        "maximal": len(find_maximal(levels)),                       # maximal itemsets (single participants included)
        "longest": len(levels),                                     # size of the longest frequent itemset
    })
counts = pd.DataFrame(count_rows)
counts["total"] = counts["singles"] + counts["size_2_or_more"]      # all frequent itemsets
counts

### Closed itemsets

An itemset is **closed** if no frequent superset has the same support. `find_closed` compares every frequent itemset with its subsets one item smaller and marks a subset with the same support as not closed. Maximal ⊆ closed ⊆ frequent; the asserts check this and the closed set found by hand on the toy.

In [ ]:
def find_closed(levels):
    """Closed itemsets: frequent itemsets that have no frequent superset with the same support."""
    not_closed = set()                                              # itemsets that occur in exactly the same episodes as a bigger itemset
    for k in range(1, len(levels)):                                 # levels[k] holds the itemsets of size k + 1
        for itemset, support in levels[k].items():
            for subset in combinations(itemset, len(itemset) - 1):  # its subsets with one item less
                if levels[k - 1][subset] == support:                # the subset has the same support as this itemset
                    not_closed.add(subset)                          # so the bigger itemset hides it: not closed
    closed = set()
    for level in levels:
        for itemset in level:
            if itemset not in not_closed:                           # every frequent superset has a lower support
                closed.add(itemset)
    return closed


assert find_closed(episode_toy_levels) == {(1,), (2,), (5,), (2, 4), (1, 2, 3)}  # closed by hand on the toy

closed_all, closed_2plus, maximal_2plus = [], [], []
for minsup in MINSUPS:                                                           # the minsups where we have our own results
    closed = find_closed(results[minsup])
    maximal = find_maximal(results[minsup])
    assert maximal <= closed                                                     # every maximal itemset is closed
    closed_all.append(len(closed))
    closed_2plus.append(sum(1 for itemset in closed if len(itemset) >= 2))       # closed itemsets with at least two participants
    maximal_2plus.append(sum(1 for itemset in maximal if len(itemset) >= 2))     # maximal itemsets with at least two participants

counts["closed"] = closed_all
counts["closed_size_2_or_more"] = closed_2plus
counts["maximal_size_2_or_more"] = maximal_2plus
counts

**Plot of the itemset counts.** One line per kind of itemset (singles, maximal, closed, size 2 or more), on a log scale, with minsup falling from left to right.

In [ ]:
import matplotlib.pyplot as plt

colors = {"singles": "#1baf7a", "maximal": "#eb6834", "closed": "#eda100", "size_2_or_more": "#2a78d6"}   # one colour per series, fixed
names = {"singles": "singles (size 1)", "maximal": "maximal itemsets", "closed": "closed itemsets", "size_2_or_more": "frequent itemsets of size >= 2"}
markers = {"singles": "s", "maximal": "D", "closed": "^", "size_2_or_more": "o"}                          # different shapes, so colour is not the only cue

fig, ax = plt.subplots(figsize=(9, 5))
for column in ["singles", "maximal", "closed", "size_2_or_more"]:
    ax.plot(counts["minsup"], counts[column], color=colors[column], marker=markers[column], lw=2, label=names[column])
    ax.annotate(f"{counts[column].iloc[-1]:,}", (counts["minsup"].iloc[-1], counts[column].iloc[-1]),
                textcoords="offset points", xytext=(8, 0), va="center")                    # value at the lowest minsup

ax.set_yscale("log")                                                                       # the counts grow by orders of magnitude
ax.invert_xaxis()                                                                          # minsup falls from left to right
ax.set_xticks(counts["minsup"])
ax.set_xticklabels([f"{m}\n{s}%" for m, s in zip(counts["minsup"], counts["share_pct"])])  # count and share of all episodes
ax.set_xlabel("minsup (number of episodes, and share of all episodes)")
ax.set_ylabel("number of itemsets (log scale)")
ax.set_title("Frequent itemsets as minsup falls")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.show()

**Reading the plot**

- Minsup falls from left to right; the y axis is logarithmic; the labels on the right are the counts at minsup 100.
- Singles grow slowly. Itemsets of two or more grow fast and overtake the singles between minsup 200 and 150.
- Closed equals frequent down to minsup 250 and separates below 200; maximal is always lower. Both include the singles.

## 10. Findings and choice of minsup

**Findings**

- From minsup 400 to 100, frequent itemsets grow from 106 to 9,812 (itemsets of two or more from 9 to 9,254, singles from 97 to 558), and the longest itemset from 3 to 10 students.
- Itemsets of two or more overtake the singles between minsup 200 (343 against 393) and 150 (1,559 against 484).
- Maximal itemsets are 86% of the frequent ones at minsup 400 (91 of 106) and 20% at minsup 100 (1,939 of 9,812).
- Closed equals frequent from minsup 400 to 250; it is 735 of 736 at 200, 1,973 of 2,043 at 150 and 8,352 of 9,812 at 100.

**Chosen: minsup 200 episodes** (0.89% of 22,530), with closed itemsets as the output.

- 736 frequent itemsets, 343 of them with at least two students and the longest with 6: an output that can be read.
- The closed itemsets keep 735 of 736 itemsets and all supports, which confidence and lift need.

**Rejected**

- Minsup 400 and 300: only 9 and 46 itemsets of two or more.
- Minsup 150: 1,559 itemsets of two or more, too many to inspect and more exposed to chance co-occurrence. A possible sensitivity check, not run.
- Minsup 100: 9,254 itemsets of two or more, unreadable without condensing.
- Maximal itemsets as the output: smaller (413 at minsup 200), but the supports of the removed itemsets are lost.

**From here on** the itemsets come from mlxtend `apriori`: the results are identical (section 8) and it is faster (0.8 s against 8.6 s at minsup 100). The rules are written from scratch.